# NVIDIA RAPIDS cuDF vs pandas — Speed Benchmark

Compares processing speed of **pandas** and **NVIDIA RAPIDS cuDF** on the
existing smart water sensor dataset.

## What this notebook benchmarks
| Operation | Why it matters |
|---|---|
| CSV / SQLite load | Ingestion throughput |
| Column-wise statistics (mean, std, min, max) | Sensor health checks |
| Boolean filtering | Anomaly isolation |
| Rolling window (60-row) | Trend smoothing for dashboard |
| GroupBy aggregation | Zone-level summary |
| Merge / join | Joining forecast with actuals |

## Requirements
- pandas (already installed)
- cuDF — requires an **NVIDIA GPU + CUDA 11.x/12.x**
  ```
  pip install cudf-cu12 --extra-index-url https://pypi.nvidia.com
  ```
  If no GPU is available, cuDF cells are skipped and the notebook
  produces a pandas-only timing report.

> **No existing files are modified.** The notebook reads from
> `database/water.db` (read-only) or from a generated synthetic CSV
> if the DB is empty.

In [ ]:
import os
import sys
import time
import sqlite3
import warnings
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── cuDF availability check ──────────────────────────────────────────────────
try:
    import cudf
    CUDF_AVAILABLE = True
    print(f'✅ cuDF {cudf.__version__} available — GPU benchmarks enabled.')
except ImportError:
    CUDF_AVAILABLE = False
    print('⚠️  cuDF not installed — running pandas-only mode.')
    print('   To enable GPU: pip install cudf-cu12 --extra-index-url https://pypi.nvidia.com')

# ── Path setup ───────────────────────────────────────────────────────────────
# Works whether run from project root or from benchmarks/
ROOT = Path(os.getcwd())
if ROOT.name == 'benchmarks':
    ROOT = ROOT.parent
DB_PATH  = ROOT / 'database' / 'water.db'
CSV_PATH = ROOT / 'benchmarks' / '_sensor_benchmark_data.csv'

print(f'Project root : {ROOT}')
print(f'DB path      : {DB_PATH}')

## 1. Load / Generate Data

In [ ]:
def load_from_sqlite(db_path: Path, limit: int = 500_000) -> pd.DataFrame:
    """Load sensor_data from the existing SQLite DB (read-only)."""
    conn = sqlite3.connect(str(db_path))
    df = pd.read_sql_query(
        f'SELECT id, timestamp, flow, pressure, ph, turbidity, temperature '
        f'FROM sensor_data ORDER BY id ASC LIMIT {limit}',
        conn
    )
    conn.close()
    return df


def generate_synthetic(n_rows: int = 200_000, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic sensor data matching the production schema."""
    rng = np.random.default_rng(seed)
    n   = n_rows

    # Inject ~5% anomalies
    anomaly_mask = rng.random(n) < 0.05

    flow        = np.where(anomaly_mask,
                           rng.uniform(1, 7, n),
                           rng.normal(15, 1.2, n))
    pressure    = np.where(anomaly_mask,
                           rng.uniform(0.5, 1.4, n),
                           rng.normal(2.5, 0.2, n))
    ph          = np.where(anomaly_mask,
                           rng.uniform(9.5, 12, n),
                           rng.normal(7.2, 0.3, n))
    turbidity   = np.where(anomaly_mask,
                           rng.uniform(6, 15, n),
                           rng.normal(2.0, 0.5, n))
    temperature = rng.normal(25.0, 1.0, n)

    timestamps = pd.date_range('2026-01-01', periods=n, freq='5s')

    return pd.DataFrame({
        'id':          np.arange(1, n + 1),
        'timestamp':   timestamps.astype(str),
        'flow':        np.maximum(0, flow),
        'pressure':    np.maximum(0, pressure),
        'ph':          np.clip(ph, 0, 14),
        'turbidity':   np.maximum(0, turbidity),
        'temperature': temperature,
    })


# ── Load or generate ──────────────────────────────────────────────────────────
TARGET_ROWS = 200_000

if DB_PATH.exists():
    df_source = load_from_sqlite(DB_PATH, limit=TARGET_ROWS)
    print(f'Loaded {len(df_source):,} rows from SQLite DB.')
    if len(df_source) < 1000:
        print(f'  Only {len(df_source)} rows in DB — padding with synthetic data.')
        df_synth  = generate_synthetic(TARGET_ROWS - len(df_source))
        df_synth['id'] += df_source['id'].max()
        df_source = pd.concat([df_source, df_synth], ignore_index=True)
else:
    print('DB not found — using fully synthetic data.')
    df_source = generate_synthetic(TARGET_ROWS)

# Save a CSV for the CSV-load benchmark
CSV_PATH.parent.mkdir(exist_ok=True)
df_source.to_csv(CSV_PATH, index=False)

print(f'Benchmark dataset: {len(df_source):,} rows  ×  {df_source.shape[1]} cols')
df_source.head(3)

## 2. Benchmark Harness

In [ ]:
import timeit
from typing import Callable

REPEAT = 5   # number of timed repetitions per benchmark

results = []  # accumulate all benchmark rows here


def bench(label: str, fn: Callable, library: str, repeat: int = REPEAT) -> float:
    """Time `fn`, record result, return median seconds."""
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    median_s = float(np.median(times))
    results.append({'operation': label, 'library': library,
                    'median_s': median_s, 'min_s': min(times), 'max_s': max(times)})
    print(f'  [{library:6s}] {label:40s}  {median_s*1000:8.2f} ms')
    return median_s


print(f'Benchmarking {len(df_source):,} rows, {REPEAT} repeats each\n')

## 3. Operation 1 — CSV Load

In [ ]:
print('=== CSV Load ===')

bench('CSV load', lambda: pd.read_csv(CSV_PATH), 'pandas')

if CUDF_AVAILABLE:
    bench('CSV load', lambda: cudf.read_csv(CSV_PATH), 'cuDF')

## 4. Operation 2 — Column Statistics

In [ ]:
print('=== Column Statistics (mean / std / min / max on 5 columns) ===')

COLS = ['flow', 'pressure', 'ph', 'turbidity', 'temperature']
pdf  = df_source.copy()

bench('Column stats', lambda: pdf[COLS].agg(['mean','std','min','max']), 'pandas')

if CUDF_AVAILABLE:
    gdf = cudf.from_pandas(pdf)
    bench('Column stats', lambda: gdf[COLS].agg(['mean','std','min','max']), 'cuDF')

## 5. Operation 3 — Boolean Filtering (Anomaly Isolation)

In [ ]:
print('=== Boolean Filter (flow<8 OR turbidity>5 OR ph outside 6.5-8.5) ===')

def pandas_filter():
    return pdf[(pdf['flow'] < 8.0) |
               (pdf['turbidity'] > 5.0) |
               (pdf['ph'] < 6.5) |
               (pdf['ph'] > 8.5)]

t_pd = bench('Boolean filter', pandas_filter, 'pandas')
n_anomalies = len(pandas_filter())
print(f'   → {n_anomalies:,} anomalous rows found ({n_anomalies/len(pdf)*100:.1f}%)')

if CUDF_AVAILABLE:
    def cudf_filter():
        return gdf[(gdf['flow'] < 8.0) |
                   (gdf['turbidity'] > 5.0) |
                   (gdf['ph'] < 6.5) |
                   (gdf['ph'] > 8.5)]
    bench('Boolean filter', cudf_filter, 'cuDF')

## 6. Operation 4 — Rolling Window (Trend Smoothing)

In [ ]:
print('=== Rolling Mean (window=60) on flow column ===')

bench('Rolling mean (w=60)', lambda: pdf['flow'].rolling(60).mean(), 'pandas')

if CUDF_AVAILABLE:
    bench('Rolling mean (w=60)', lambda: gdf['flow'].rolling(60).mean(), 'cuDF')

## 7. Operation 5 — GroupBy Aggregation (Zone-Level Summary)

In [ ]:
print('=== GroupBy aggregation (synthetic zone bucket × 5 metrics) ===')

# Create a synthetic zone column (6 zones)
pdf_z = pdf.copy()
pdf_z['zone'] = (pdf_z['id'] % 6).astype(str)

def pandas_groupby():
    return pdf_z.groupby('zone')[COLS].agg(['mean', 'std', 'min', 'max'])

bench('GroupBy (6 zones, 5 cols)', pandas_groupby, 'pandas')

if CUDF_AVAILABLE:
    gdf_z = gdf.copy()
    gdf_z['zone'] = (gdf_z['id'] % 6).astype(str)
    def cudf_groupby():
        return gdf_z.groupby('zone')[COLS].agg(['mean', 'std', 'min', 'max'])
    bench('GroupBy (6 zones, 5 cols)', cudf_groupby, 'cuDF')

## 8. Operation 6 — Merge / Join (Actuals vs Forecast)

In [ ]:
print('=== Merge (inner join on id, simulating actuals ↔ forecast) ===')

# Simulate a forecast DataFrame with every 12th row (5-min aggregates)
forecast_ids = pdf['id'].values[::12]
pdf_forecast = pd.DataFrame({
    'id':             forecast_ids,
    'forecast_flow':  np.random.normal(15, 1, len(forecast_ids)),
})

bench('Inner merge', lambda: pdf.merge(pdf_forecast, on='id', how='inner'), 'pandas')

if CUDF_AVAILABLE:
    gdf_forecast = cudf.from_pandas(pdf_forecast)
    bench('Inner merge', lambda: gdf.merge(gdf_forecast, on='id', how='inner'), 'cuDF')

## 9. Results Summary

In [ ]:
import matplotlib
matplotlib.use('Agg')   # headless-safe
import matplotlib.pyplot as plt

df_results = pd.DataFrame(results)
df_results['median_ms'] = df_results['median_s'] * 1000

# ── Pivot table ───────────────────────────────────────────────────────────────
pivot = df_results.pivot(index='operation', columns='library', values='median_ms').round(2)

if CUDF_AVAILABLE and 'cuDF' in pivot.columns and 'pandas' in pivot.columns:
    pivot['Speedup (×)'] = (pivot['pandas'] / pivot['cuDF']).round(1)

print('\n' + '='*70)
print('BENCHMARK RESULTS  (median over', REPEAT, 'runs)')
print('='*70)
print(pivot.to_string())
print('='*70)
print(f'Dataset: {len(df_source):,} rows')

if CUDF_AVAILABLE and 'Speedup (×)' in pivot.columns:
    mean_speedup = pivot['Speedup (×)'].mean()
    print(f'Mean GPU speedup across all operations: {mean_speedup:.1f}×')
else:
    print('(cuDF not available — pandas-only results shown)')

pivot

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'NVIDIA RAPIDS cuDF vs pandas  ·  {len(df_source):,} rows',
    fontsize=14, fontweight='bold'
)

# Left: absolute times
ax = axes[0]
ops = pivot.index.tolist()
x   = np.arange(len(ops))
w   = 0.35

pd_times = pivot.get('pandas', pd.Series([0]*len(ops))).values
ax.bar(x - w/2, pd_times, w, label='pandas', color='#3b82f6', alpha=0.85)

if CUDF_AVAILABLE and 'cuDF' in pivot.columns:
    cu_times = pivot['cuDF'].values
    ax.bar(x + w/2, cu_times, w, label='cuDF (GPU)', color='#10b981', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(
    [o[:22] for o in ops], rotation=35, ha='right', fontsize=9
)
ax.set_ylabel('Median time (ms)')
ax.set_title('Absolute execution time (lower = better)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Right: speedup bars (only meaningful with cuDF)
ax2 = axes[1]
if CUDF_AVAILABLE and 'Speedup (×)' in pivot.columns:
    speedups = pivot['Speedup (×)'].values
    colours  = ['#10b981' if s >= 1 else '#dc2626' for s in speedups]
    ax2.bar(x, speedups, color=colours, alpha=0.85)
    ax2.axhline(1.0, color='gray', linewidth=1.5, linestyle='--', label='1× (no speedup)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(
        [o[:22] for o in ops], rotation=35, ha='right', fontsize=9
    )
    ax2.set_ylabel('Speedup (pandas time / cuDF time)')
    ax2.set_title('GPU speedup over pandas (higher = better)')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    for i, s in enumerate(speedups):
        ax2.text(i, s + 0.05, f'{s:.1f}×', ha='center', fontsize=9, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'cuDF not available\nInstall NVIDIA RAPIDS to see speedup',
             transform=ax2.transAxes, ha='center', va='center',
             fontsize=12, color='gray')
    ax2.set_title('GPU speedup (unavailable)')

plt.tight_layout()
chart_path = ROOT / 'benchmarks' / 'rapids_vs_pandas_results.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {chart_path}')

## 10. Interpretation & Hackathon Talking Points

### When cuDF wins
- **CSV load** and **boolean filtering** on large datasets typically show
  5–20× GPU speedup because the GPU processes many columns in parallel.
- **GroupBy aggregations** benefit most on wide, high-cardinality datasets.
- **Rolling windows** on long time-series gain 3–10× depending on window size.

### When pandas is competitive
- Very small datasets (< 50 k rows) — GPU transfer overhead dominates.
- Simple scalar operations — Python overhead masks GPU benefit.

### Relevance to this system
In a production deployment of the Smart Water Management System:
- A city network could generate **10 M+ readings/day** across hundreds of zones.
- Replacing the pandas-based anomaly batch scan with cuDF would allow
  **real-time city-wide analysis** instead of batched per-zone scans.
- The LSTM feature extraction pipeline (rolling stats → model input)
  would be **substantially faster**, enabling sub-second alert latency.

### How to enable RAPIDS in production
```python
# Swap pandas for cuDF with a single import alias (zero code changes)
try:
    import cudf as pd
except ImportError:
    import pandas as pd
```
Because cuDF mirrors the pandas API, the existing `anomaly_detection.py`
and `lstm_forecast.py` would work without modification.

In [ ]:
# Cleanup: remove temporary CSV after benchmarking
if CSV_PATH.exists():
    CSV_PATH.unlink()
    print(f'Removed temporary file: {CSV_PATH}')
print('Benchmark complete.')